# N-Gram + BPC Final Notebook

This notebook is intended to generate a **final-ready** export for the report.

It runs:

- `word`, `char`, `bpe`
- `1-gram`, `2-gram`, `3-gram`
- BPC as the main cross-tokenizer metric
- PPL as a complementary metric
- raw-text next-token prediction examples for all tokenizers
- optional sentence-scoring examples

Compared with earlier versions, this notebook now uses **character-based split limits** so each tokenizer is evaluated on the same amount of original text.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from typing import Optional

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git"
GIT_REF = "bpc-metric"
REPO_NAME = "text-preprocess-tokenization"
REPO_DIR = None  # e.g. "/root/text-preprocess-tokenization" if the repo already exists
AUTO_CLONE_IF_MISSING = True
AUTO_PULL_LATEST = False


def looks_like_repo_root(path: Path) -> bool:
    return (path / "requirements.txt").exists() and (path / "src").exists() and (path / "main.py").exists()


def find_repo_root(start: Path) -> Optional[Path]:
    common_roots = [start, *start.parents, Path.home(), Path("/root"), Path("/content"), Path("/workspace"), Path("/mnt"), Path("/tmp")]
    seen = set()
    candidates = []
    for root in common_roots:
        if not root.exists():
            continue
        candidates.append(root)
        candidates.append(root / REPO_NAME)
        try:
            for child in root.iterdir():
                if child.is_dir() and child.name == REPO_NAME:
                    candidates.append(child)
        except OSError:
            pass

    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if looks_like_repo_root(candidate):
            return candidate
    return None


def run_git(*args: str) -> None:
    subprocess.run(["git", *args], check=True)


if REPO_DIR is not None:
    project_root = Path(REPO_DIR).expanduser().resolve()
    if not looks_like_repo_root(project_root):
        raise FileNotFoundError(f"REPO_DIR does not look like the repo root: {project_root}")
else:
    detected_root = find_repo_root(Path.cwd().resolve())
    if detected_root is None and AUTO_CLONE_IF_MISSING:
        clone_parent = Path("/content") if Path("/content").exists() else Path.home()
        project_root = (clone_parent / REPO_NAME).resolve()
        if not project_root.exists():
            run_git("clone", REPO_URL, str(project_root))
        run_git("-C", str(project_root), "checkout", GIT_REF)
    elif detected_root is None:
        raise FileNotFoundError(
            "Could not find the project root automatically. "
            "Set REPO_DIR to the repo path, or allow AUTO_CLONE_IF_MISSING."
        )
    else:
        project_root = detected_root

if AUTO_PULL_LATEST:
    run_git("-C", str(project_root), "fetch", "origin")
    run_git("-C", str(project_root), "checkout", GIT_REF)
    run_git("-C", str(project_root), "pull", "--ff-only", "origin", GIT_REF)

os.chdir(project_root)
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python import root added: {PROJECT_ROOT}")
print(f"Using branch/config target: {GIT_REF}")

Cloning into '/root/text-preprocess-tokenization'...


branch 'bpc-metric' set up to track 'origin/bpc-metric'.
Project root: /root/text-preprocess-tokenization
Python import root added: /root/text-preprocess-tokenization
Using branch/config target: bpc-metric


Switched to a new branch 'bpc-metric'


In [2]:
!pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import json
import shutil

import pandas as pd

from src.datasets.load_data import load
from src.training.train_ngram import NGramTrainingConfig, train_ngram_language_model

METRICS_ROOT = PROJECT_ROOT / "outputs" / "metrics" / "ngram"
ARTIFACT_ROOT = PROJECT_ROOT / "outputs" / "artifacts" / "ngram"
REPORT_ROOT = PROJECT_ROOT / "outputs" / "report_tables" / "ngram"


def load_metrics(run_name: str) -> dict:
    return json.loads((METRICS_ROOT / f"{run_name}.json").read_text(encoding="utf-8"))


def make_run_name(dataset_name: str, tokenizer_name: str, order: int, profile: str) -> str:
    return f"report_{dataset_name.replace('-', '_')}_{tokenizer_name}_{order}gram_{profile}_bpc"


def build_comparison_row(metrics: dict) -> dict:
    train_split = metrics["splits"]["train"]
    validation_split = metrics["splits"]["validation"]
    test_split = metrics["splits"]["test"]
    return {
        "dataset": metrics["config"]["dataset_name"],
        "tokenizer": metrics["tokenizer"]["type"],
        "n_gram": f"{metrics['model']['order']}-gram",
        "order": metrics["model"]["order"],
        "vocab_size": metrics["tokenizer"]["vocab_size"],
        "train_tokens": train_split["num_tokens"],
        "train_characters": train_split["num_characters"],
        "validation_characters": validation_split["num_characters"],
        "test_characters": test_split["num_characters"],
        "tokenizer_fit_s": round(metrics["timing"]["tokenizer_fit_seconds"], 4),
        "model_fit_s": round(metrics["timing"]["model_fit_seconds"], 4),
        "total_s": round(metrics["timing"]["total_seconds"], 4),
        "val_bpc": round(validation_split["bits_per_character"], 4),
        "test_bpc": round(test_split["bits_per_character"], 4),
        "val_avg_nll": round(validation_split["average_negative_log_likelihood"], 4),
        "test_avg_nll": round(test_split["average_negative_log_likelihood"], 4),
        "val_ppl": round(validation_split["perplexity"], 4),
        "test_ppl": round(test_split["perplexity"], 4),
        "run_name": metrics["run_name"],
    }


def visible_context(text: str) -> str:
    return text.replace(" ", "<sp>")


def build_prediction_rows(metrics: dict) -> list[dict]:
    rows = []
    for item in metrics["prediction_contexts"]:
        top_predictions = ", ".join(
            f"{pred['token']} ({pred['probability']:.4g})" for pred in item["predictions"]
        )
        top_1_prediction = item["predictions"][0]["token"] if item["predictions"] else ""
        rows.append(
            {
                "dataset": metrics["config"]["dataset_name"],
                "tokenizer": metrics["tokenizer"]["type"],
                "n_gram": f"{metrics['model']['order']}-gram",
                "context": item["context_text"],
                "context_visible": visible_context(item["context_text"]),
                "top_1_prediction": top_1_prediction,
                "top_predictions": top_predictions,
            }
        )
    return rows


def build_scored_text_rows(metrics: dict) -> list[dict]:
    rows = []
    for item in metrics["scored_texts"]:
        rows.append(
            {
                "dataset": metrics["config"]["dataset_name"],
                "tokenizer": metrics["tokenizer"]["type"],
                "n_gram": f"{metrics['model']['order']}-gram",
                "text": item["text"],
                "avg_nll": round(item["average_negative_log_likelihood"], 4),
                "bpc": round(item["bits_per_character"], 4),
                "ppl": round(item["perplexity"], 4),
            }
        )
    return rows


## Experiment configuration

Recommended defaults:

- `PROFILE = "medium"` for report-oriented runs
- `TOKENIZER_NAMES = ["word", "char", "bpe"]`
- `NGRAM_ORDERS = [1, 2, 3]`

This notebook uses **character-based split limits** in the experiment profiles below, so each tokenizer sees the same amount of original text per split.

The raw-text prediction contexts below intentionally keep trailing spaces. This makes the qualitative next-token examples more aligned across tokenizers:

- word/BPE predict the next lexical unit
- char predicts the next character in the same raw continuation

In [4]:
DATASET_NAME = "one-billion-word"
PROFILE = "medium"  # quick | medium | full

TOKENIZER_NAMES = ["word", "char", "bpe"]
NGRAM_ORDERS = [1, 2, 3]
LAPLACE_ALPHA = 1.0
TOP_K = 5

MAX_VOCAB_SIZE_BY_TOKENIZER = {
    "word": 50_000,
    "char": None,
    "bpe": 50_000,
}

PREDICTION_CONTEXTS = [
    "the history ",
    "united ",
    "world war ",
]

# Optional qualitative sentence-scoring examples.
# Leave this empty if you want the notebook to focus on next-token prediction only.
SCORE_TEXTS = []

PROFILES = {
    "quick": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "wikitext-103": {
            "max_fit_texts": 500,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "one-billion-word": {
            "max_fit_texts": 500,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
    },
    "medium": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "wikitext-103": {
            "max_fit_texts": 2_000,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "one-billion-word": {
            "max_fit_texts": 2_000,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
    },
    "full": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "wikitext-103": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "one-billion-word": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
    },
}

limits = PROFILES[PROFILE][DATASET_NAME].copy()
print({
    "dataset": DATASET_NAME,
    "profile": PROFILE,
    "tokenizers": TOKENIZER_NAMES,
    "orders": NGRAM_ORDERS,
    **limits,
})

{'dataset': 'one-billion-word', 'profile': 'medium', 'tokenizers': ['word', 'char', 'bpe'], 'orders': [1, 2, 3], 'max_fit_texts': 2000, 'max_fit_characters': 1000000, 'max_train_tokens': None, 'max_train_characters': 1000000, 'max_validation_tokens': None, 'max_validation_characters': 250000, 'max_test_tokens': None, 'max_test_characters': 250000}


In [5]:
load(DATASET_NAME)
print(f"Dataset ready: {DATASET_NAME}")

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/9 shards):   0%|          | 0/30301028 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/306688 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/306688 [00:00<?, ? examples/s]

Dataset ready: one-billion-word


In [6]:
comparison_rows = []
prediction_rows = []
scored_text_rows = []
run_names = []

for tokenizer_name in TOKENIZER_NAMES:
    for order in NGRAM_ORDERS:
        run_name = make_run_name(DATASET_NAME, tokenizer_name, order, PROFILE)
        print(f"Running {run_name} ...")

        config = NGramTrainingConfig(
            dataset_name=DATASET_NAME,
            tokenizer_name=tokenizer_name,
            order=order,
            alpha=LAPLACE_ALPHA,
            max_vocab_size=MAX_VOCAB_SIZE_BY_TOKENIZER[tokenizer_name],
            max_fit_texts=limits["max_fit_texts"],
            max_fit_characters=limits["max_fit_characters"],
            max_train_tokens=limits["max_train_tokens"],
            max_train_characters=limits["max_train_characters"],
            max_validation_tokens=limits["max_validation_tokens"],
            max_validation_characters=limits["max_validation_characters"],
            max_test_tokens=limits["max_test_tokens"],
            max_test_characters=limits["max_test_characters"],
            run_name=run_name,
        )

        summary = train_ngram_language_model(
            config,
            prediction_contexts=PREDICTION_CONTEXTS,
            score_texts=SCORE_TEXTS,
            top_k=TOP_K,
        )

        metrics = load_metrics(summary["run_name"])
        run_names.append(summary["run_name"])
        comparison_rows.append(build_comparison_row(metrics))
        prediction_rows.extend(build_prediction_rows(metrics))
        scored_text_rows.extend(build_scored_text_rows(metrics))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["tokenizer"] = pd.Categorical(comparison_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
comparison_df = comparison_df.sort_values(["tokenizer", "order"]).reset_index(drop=True)

report_df = comparison_df[[
    "tokenizer",
    "n_gram",
    "train_tokens",
    "train_characters",
    "tokenizer_fit_s",
    "model_fit_s",
    "val_bpc",
    "val_avg_nll",
    "val_ppl",
    "test_bpc",
    "test_avg_nll",
    "test_ppl",
]]

display(report_df)

Running report_one_billion_word_word_1gram_medium_bpc ...


Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Saving the dataset (0/9 shards):   0%|          | 0/30301028 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/306688 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/306688 [00:00<?, ? examples/s]

Prepared n-gram run with dataset=one-billion-word, tokenizer=word, order=1, vocab_size=10940
  train ppl 519.3781 | train bpc 1.7506 | val ppl 395.9045 | val bpc 1.6717 | test ppl 395.9045 | test bpc 1.6717
Top 5 predictions for context: 'the history '
  'the': 0.043403
  ',': 0.041657
  '.': 0.035598
  'to': 0.021701
  'of': 0.020916
Top 5 predictions for context: 'united '
  'the': 0.043403
  ',': 0.041657
  '.': 0.035598
  'to': 0.021701
  'of': 0.020916
Top 5 predictions for context: 'world war '
  'the': 0.043403
  ',': 0.041657
  '.': 0.035598
  'to': 0.021701
  'of': 0.020916
Saved artifacts to /root/text-preprocess-tokenization/outputs/artifacts/ngram/report_one_billion_word_word_1gram_medium_bpc
Saved metrics to /root/text-preprocess-tokenization/outputs/metrics/ngram/report_one_billion_word_word_1gram_medium_bpc.json
Running report_one_billion_word_word_2gram_medium_bpc ...
Prepared n-gram run with dataset=one-billion-word, tokenizer=word, order=2, vocab_size=10940
  train pp

,tokenizer,n_gram,train_tokens,train_characters,tokenizer_fit_s,model_fit_s,val_bpc,val_avg_nll,val_ppl,test_bpc,test_avg_nll,test_ppl
0,word,1-gram,194070,1000000,0.0522,0.2295,1.6717,5.9812,395.9045,1.6717,5.9812,395.9045
1,word,2-gram,194070,1000000,0.0525,0.4890,1.9524,6.9853,1080.6404,1.9524,6.9853,1080.6404
2,word,3-gram,194070,1000000,0.0529,1.1476,2.3598,8.4429,4641.9847,2.3598,8.4429,4641.9847
3,char,1-gram,1007344,1000000,0.0631,1.1250,4.5538,3.1342,22.9697,4.5538,3.1342,22.9697
4,char,2-gram,1007344,1000000,0.0625,2.1798,3.5945,2.4739,11.8686,3.5945,2.4739,11.8686
5,char,3-gram,1007344,1000000,0.0625,3.2614,3.0877,2.1251,8.3740,3.0877,2.1251,8.3740
6,bpe,1-gram,223208,1000000,20.7264,0.1927,2.4315,7.2198,1366.2157,2.4315,7.2198,1366.2157
7,bpe,2-gram,223208,1000000,18.3420,0.4347,2.8325,8.4104,4493.7412,2.8325,8.4104,4493.7412
8,bpe,3-gram,223208,1000000,18.1004,1.1848,3.1794,9.4403,12585.9101,3.1794,9.4403,12585.9101


In [7]:
prediction_df = pd.DataFrame(prediction_rows)
if not prediction_df.empty:
    prediction_df["tokenizer"] = pd.Categorical(prediction_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
    prediction_df = prediction_df.sort_values(["tokenizer", "n_gram", "context"]).reset_index(drop=True)

scored_text_df = pd.DataFrame(scored_text_rows)
if not scored_text_df.empty:
    scored_text_df["tokenizer"] = pd.Categorical(scored_text_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
    scored_text_df = scored_text_df.sort_values(["tokenizer", "n_gram", "text"]).reset_index(drop=True)

print("Raw-text next-token prediction examples")
display(prediction_df)

print("Optional sentence scoring examples")
display(scored_text_df)

Raw-text next-token prediction examples


,dataset,tokenizer,n_gram,context,context_visible,top_1_prediction,top_predictions
0,one-billion-word,word,1-gram,the history,the<sp>history<sp>,the,"the (0.0434), , (0.04166), . (0.0356), to (0.0..."
1,one-billion-word,word,1-gram,united,united<sp>,the,"the (0.0434), , (0.04166), . (0.0356), to (0.0..."
2,one-billion-word,word,1-gram,world war,world<sp>war<sp>,the,"the (0.0434), , (0.04166), . (0.0356), to (0.0..."
3,one-billion-word,word,2-gram,the history,the<sp>history<sp>,of,"of (0.0008198), . (0.0007287), , (0.0006376), ..."
4,one-billion-word,word,2-gram,united,united<sp>,under,"under (0.0001828), Europe (0.0001828), , (9.13..."
5,one-billion-word,word,2-gram,world war,world<sp>war<sp>,in,"in (0.0007283), , (0.0005462), . (0.0005462), ..."
6,one-billion-word,word,3-gram,the history,the<sp>history<sp>,of,"of (0.0003655), , (9.138e-05), the (9.138e-05)..."
7,one-billion-word,word,3-gram,united,united<sp>,under,"under (0.0001828), Europe (0.0001828), , (9.13..."
8,one-billion-word,word,3-gram,world war,world<sp>war<sp>,.,". (0.0001828), two (0.0001828), , (9.139e-05),..."
9,one-billion-word,char,1-gram,the history,the<sp>history<sp>,,"(0.1781), e (0.09288), t (0.06579), a (0.064..."


Optional sentence scoring examples


""


In [8]:
export_dir = REPORT_ROOT / DATASET_NAME / PROFILE
zip_base = PROJECT_ROOT / f"report_export_{DATASET_NAME.replace('-', '_')}_{PROFILE}_ngram_bpc"

if export_dir.exists():
    shutil.rmtree(export_dir)

(export_dir / "metrics").mkdir(parents=True, exist_ok=True)
(export_dir / "artifacts").mkdir(parents=True, exist_ok=True)

report_df.to_csv(export_dir / "quantitative_comparison.csv", index=False)
prediction_df.to_csv(export_dir / "prediction_examples.csv", index=False)
scored_text_df.to_csv(export_dir / "sentence_scoring_examples.csv", index=False)

(export_dir / "run_names.json").write_text(json.dumps(run_names, indent=2), encoding="utf-8")
(export_dir / "config.json").write_text(
    json.dumps(
        {
            "dataset": DATASET_NAME,
            "profile": PROFILE,
            "tokenizers": TOKENIZER_NAMES,
            "orders": NGRAM_ORDERS,
            "limits": limits,
            "max_vocab_size_by_tokenizer": MAX_VOCAB_SIZE_BY_TOKENIZER,
            "prediction_contexts": PREDICTION_CONTEXTS,
            "score_texts": SCORE_TEXTS,
        },
        indent=2,
    ),
    encoding="utf-8",
)

for run_name in run_names:
    shutil.copy2(METRICS_ROOT / f"{run_name}.json", export_dir / "metrics" / f"{run_name}.json")
    shutil.copytree(ARTIFACT_ROOT / run_name, export_dir / "artifacts" / run_name, dirs_exist_ok=True)

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=export_dir)
print("Export directory:", export_dir)
print("Created zip:", zip_path)

Export directory: /root/text-preprocess-tokenization/outputs/report_tables/ngram/one-billion-word/medium
Created zip: /root/text-preprocess-tokenization/report_export_one_billion_word_medium_ngram_bpc.zip


In [9]:
if IS_COLAB:
    from google.colab import files
    files.download(str(Path(f"{zip_base}.zip")))
else:
    print(Path(f"{zip_base}.zip"))

/root/text-preprocess-tokenization/report_export_one_billion_word_medium_ngram_bpc.zip
